# Music Annotation Project V8.7
## Notebook 51: Frozen M3 vs M5-REMOVE vs diagnostic M4 Reconstruction Crossover

**Runtime:** GPU recommended because the frozen learned U-Net restorer is included.

### Scientific question
Does the frozen REMOVE-only M5 mask from Notebook 50 make reconstruction better than frozen M3 across unchanged restorers, and does better mask conditioning reduce sensitivity to restorer choice?

### Fixed design
- Mask arms: `M3`, `M5_REMOVE`, `M4_DIAGNOSTIC`.
- Restorers: `R1_biharmonic`, `R2_telea`, `R3_learned_unet`.
- No SAFE_PROTECT intervention.
- `M5_REMOVE = M3 & ~SAFE_REMOVE` exactly.
- M4 is diagnostic only and has no promotion authority.
- Clean truth is scoring-only. It is loaded only after all candidate outputs and hashes are written.
- Outside-authoritative-mask bytes are immutable.
- Page/source-group paired analysis is mandatory.
- Restorer-gap contraction is a registered secondary endpoint, not a substitute for reconstruction quality.

This notebook does not retrain Notebook 50, retune its threshold, or tune any restorer.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-image torch torchvision joblib


Mounted at /content/drive


## 1. Configuration and pre-registration


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, random, sys, platform, time, math, gc
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import rankdata
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import inpaint as sk_inpaint
import torch
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N50_ROOT=ROOT/'observable_residual_mask_m5_v8_7'
N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
N46_ROOT=ROOT/'observed_print_classifier_v8_6_notebook46'
OUT_ROOT=ROOT/'residual_mask_crossover_v8_7_notebook51'
N50_RUN_OVERRIDE=None; N45_RUN_OVERRIDE=None; N46_RUN_OVERRIDE=None
SEED=51001; BOOTSTRAP=10000; CROP_MARGIN=48
MASK_ARMS=['M3','M5_REMOVE','M4_DIAGNOSTIC']
RESTORERS=['R1_biharmonic','R2_telea','R3_learned_unet']
PRIMARY_RESTORER='R3_learned_unet'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_7_notebook51'
OUT=OUT_ROOT/RUN; INPUT=OUT/'01_source_lock'; CHECK=OUT/'02_page_checkpoints'; CAND=OUT/'03_candidates'; LOCK=OUT/'04_candidate_lock'; TABLES=OUT/'05_tables'; VIS=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUT,CHECK,CAND,LOCK,TABLES,VIS,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(p,v):
 p=Path(p); q=Path(str(p)+'.tmp'); q.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); q.replace(p)
def require(x,msg):
 if not x: atomic_json(AUDIT/'fatal_failure.json',{'utc':datetime.now(timezone.utc).isoformat(),'message':msg}); raise RuntimeError(msg)
PREREG={'version':'n51-v1','development_only':True,'mask_arms':MASK_ARMS,'restorers':RESTORERS,'primary_restorer':PRIMARY_RESTORER,'m5_definition':'M3 minus SAFE_REMOVE only; no SAFE_PROTECT','m4_role':'diagnostic only','primary_test':'paired M5_REMOVE versus M3 reconstruction benefit','secondary_test':'restorer-gap contraction from M3 to M5_REMOVE','bootstrap':'page-clustered 10000 resamples','truth_firewall':'clean truth unavailable until candidate lock','promotion_authority':False}
atomic_json(OUT/'01_pre_registration.json',PREREG)
print('Output:',OUT,'Device:',DEVICE)


Output: /content/drive/MyDrive/MusicAnnotationProject/residual_mask_crossover_v8_7_notebook51/20261029_223357_v8_7_notebook51 Device: cpu


## 2. Resolve exact Notebook 50, Notebook 45 and Notebook 46 lineage


In [4]:
def newest(root, required):
 hits=[]
 if Path(root).exists():
  for p in Path(root).rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
N50_REQUIRED=['notebook50_verdict.json','experiment_identity.json','m5_map_inventory.csv','selected_models.json']
N50=Path(N50_RUN_OVERRIDE) if N50_RUN_OVERRIDE else newest(N50_ROOT,N50_REQUIRED)
require(N50 is not None,'No complete Notebook 50 run found. Set N50_RUN_OVERRIDE.')
v50=json.loads((N50/'notebook50_verdict.json').read_text())
require(v50.get('verdict')=='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER',f'Notebook 50 verdict does not authorise REMOVE-only crossover: {v50.get("verdict")}')
id50=json.loads((N50/'experiment_identity.json').read_text())
# Resolve N45/N46 from explicit overrides first, then normal durable roots.
N45=Path(N45_RUN_OVERRIDE) if N45_RUN_OVERRIDE else newest(N45_ROOT,['mask_lock.json','source_record.json'])
N46=Path(N46_RUN_OVERRIDE) if N46_RUN_OVERRIDE else newest(N46_ROOT,['v8_6_verdict.json','experiment_identity.json'])
require(N45 is not None,'No complete Notebook 45 run found. Set N45_RUN_OVERRIDE.')
require(N46 is not None,'No complete Notebook 46 run found. Set N46_RUN_OVERRIDE.')
v46=json.loads((N46/'v8_6_verdict.json').read_text()); require(v46.get('run_valid') is True or v46.get('valid_run') is True,'Notebook 46 is not recorded as valid.')
source={'notebook50':str(N50),'notebook45':str(N45),'notebook46':str(N46),'notebook50_verdict':v50.get('verdict'),'notebook46_verdict':v46}
atomic_json(OUT/'source_record.json',source)
for p in [N50/'notebook50_verdict.json',N50/'experiment_identity.json',N45/'mask_lock.json',N46/'v8_6_verdict.json',N46/'experiment_identity.json']:
 shutil.copy2(p,INPUT/(p.parent.name+'__'+p.name))
print(json.dumps(source,indent=2))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_6_notebook46/20261021_131744_v8_6_notebook46/experiment_identity.json'

## 3. Load frozen M3, M5-REMOVE and M4 masks

Notebook 51 never reconstructs SAFE_REMOVE from a model. It consumes Notebook 50's saved decision maps. The page-level M3/M4 masks are rebuilt from Notebook 45's regional PNGs exactly as the downstream Notebook 46 family consumed them. The M5 assertion must be a strict subset of M3 and there must be **zero SAFE_PROTECT intervention**.


In [ ]:
SUP={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUP} if Path(folder).exists() else {}
def find_page_arm(run,page,arm):
 hits=list(Path(run).glob(f'masks/{page}/**/{arm}.png'))+list(Path(run).glob(f'**/{page}/**/{arm}.png'))
 hits=list(dict.fromkeys(hits)); require(len(hits)>0,f'{page}: no Notebook45 {arm}.png')
 arr=None
 for p in hits:
  a=cv2.imread(str(p),0); require(a is not None,f'Unreadable {p}'); a=a>0; arr=a.copy() if arr is None else (arr|a)
 return arr
# Robustly locate Notebook50 per-page maps documented by its inventory.
inv=pd.read_csv(N50/'m5_map_inventory.csv'); require({'page'}.issubset(inv.columns),'Notebook50 m5_map_inventory lacks page.')
def resolve_n50_mask(page, kind):
 candidates=[]
 if kind=='SAFE_REMOVE':
  candidates += list(N50.glob(f'**/{page}/SAFE_REMOVE.png'))+list(N50.glob(f'**/{page}*safe_remove*.png'))+list(N50.glob(f'**/{page}*REMOVE*.png'))
 if kind=='M5': candidates += list(N50.glob(f'**/{page}/M5.png'))+list(N50.glob(f'**/{page}*m5*.png'))
 candidates=[p for p in candidates if p.is_file()]
 require(candidates,f'{page}: cannot resolve Notebook50 {kind} mask.')
 a=cv2.imread(str(candidates[0]),0); require(a is not None,f'{page}: unreadable {candidates[0]}'); return a>0,candidates[0]
pages=sorted(inv.page.astype(str).unique()); masks={}; mask_rows=[]
for page in pages:
 m3=find_page_arm(N45,page,'M3'); m4=find_page_arm(N45,page,'M4')
 safe_remove,prem=resolve_n50_mask(page,'SAFE_REMOVE')
 require(m3.shape==m4.shape==safe_remove.shape,f'{page}: mask shape mismatch.')
 require(not np.any(safe_remove&~m3),f'{page}: SAFE_REMOVE escapes M3.')
 m5=m3&~safe_remove
 # If Notebook50 saved M5, prove equivalence.
 try:
  saved,p5=resolve_n50_mask(page,'M5'); require(np.array_equal(saved,m5),f'{page}: saved M5 != M3 minus SAFE_REMOVE')
 except RuntimeError:
  p5=None
 masks[page]={'M3':m3,'M5_REMOVE':m5,'M4_DIAGNOSTIC':m4,'SAFE_REMOVE':safe_remove}
 mask_rows.append({'page':page,'M3':int(m3.sum()),'SAFE_REMOVE':int(safe_remove.sum()),'M5_REMOVE':int(m5.sum()),'M4_DIAGNOSTIC':int(m4.sum()),'m5_subset_m3':True,'safe_remove_path':str(prem),'saved_m5_path':str(p5) if p5 else None})
mask_df=pd.DataFrame(mask_rows); mask_df.to_csv(TABLES/'mask_arm_inventory.csv',index=False)
print(mask_df[['M3','SAFE_REMOVE','M5_REMOVE','M4_DIAGNOSTIC']].sum())


## 4. Resolve observed pages, source groups, authoritative mask and frozen learned U-Net


In [ ]:
# Page/source identity: prefer Notebook50 truth-evaluation output, otherwise Notebook45 region manifest.
meta=None
for p in [N50/'selected_region_truth_evaluation.csv',N50/'source_group_split_audit.csv',N45/'tables'/'region_manifest.parquet']:
 if p.exists():
  q=pd.read_parquet(p) if p.suffix=='.parquet' else pd.read_csv(p)
  if {'page','source_group'}.issubset(q.columns): meta=q[['page','source_group']].drop_duplicates(); break
require(meta is not None,'Cannot resolve page/source_group mapping.')
meta.page=meta.page.astype(str); require(not meta.page.duplicated().any(),'Page maps to multiple source groups.')
meta=meta.set_index('page')
observed_map=imap(ROOT/'aligned'); require(set(pages).issubset(observed_map),f'Missing aligned observed pages: {sorted(set(pages)-set(observed_map))}')
# Authoritative boundary: use frozen V7.5 cache referenced by N46 if recoverable; otherwise union M3/M4 is NOT accepted as authority.
# Find a cache_status that covers the roster.
cache_status=[]
for p in ROOT.rglob('cache_status.csv'):
 try:
  q=pd.read_csv(p)
  if {'page','cache'}.issubset(q.columns) and set(pages).issubset(set(q.page.astype(str))): cache_status.append((p,q))
 except Exception: pass
require(cache_status,'Could not resolve authoritative cache_status.csv covering Notebook51 pages.')
cache_status.sort(key=lambda x:x[0].stat().st_mtime); cache_table=cache_status[-1][1]; cache_lookup={str(r.page):Path(r.cache) for _,r in cache_table.iterrows()}
# Frozen U-Net: use Notebook46 checkpoint inventory/identity, never retrain.
def find_unet_checkpoint(run):
 c=[]
 for ext in ['*.pt','*.pth','*.joblib']:
  for p in Path(run).rglob(ext):
   if 'unet' in p.name.lower() or 'r3' in p.name.lower(): c.append(p)
 require(c,'No frozen R3 U-Net checkpoint found beneath Notebook46 run.')
 return sorted(c,key=lambda p:p.stat().st_mtime)[-1]
UNET_CKPT=find_unet_checkpoint(N46)
print('Frozen U-Net checkpoint:',UNET_CKPT)


## 5. Frozen reconstruction implementations

Telea and biharmonic use the controlled crop-only pattern and replace bytes only inside the active residual mask. The learned U-Net loader is intentionally strict: Notebook 51 will not guess a new architecture. It accepts only a TorchScript module or a serialized `nn.Module`; otherwise it stops and asks for the exact Notebook 46 model constructor to be copied in. This avoids silently evaluating a different U-Net.


In [ ]:
def padded_components(mask,margin=CROP_MARGIN,min_area=1):
 n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8); h,w=mask.shape; boxes=[]
 for i in range(1,n):
  x,y,bw,bh,a=stats[i]
  if a>=min_area: boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
 return boxes
def immutable(original,candidate,mask):
 out=original.copy(); out[mask]=candidate[mask]; require(np.array_equal(out[~mask],original[~mask]),'Restorer changed outside active residual mask.'); return out
def restore_classical(obs,mask,method):
 result=obs.copy()
 for x0,y0,x1,y1 in padded_components(mask):
  im=obs[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
  if method=='R2_telea': cand=cv2.inpaint(im,(mk*255).astype(np.uint8),3,cv2.INPAINT_TELEA)
  elif method=='R1_biharmonic': cand=np.clip(sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255.,mk,channel_axis=None)*255,0,255).astype(np.uint8)
  else: raise ValueError(method)
  result[y0:y1,x0:x1][mk]=cand[mk]
 return immutable(obs,result,mask)
def load_frozen_unet(path):
 try:
  m=torch.jit.load(str(path),map_location=DEVICE); m.eval(); return m,'torchscript'
 except Exception: pass
 try:
  obj=torch.load(path,map_location=DEVICE,weights_only=False)
  if isinstance(obj,nn.Module): obj.eval(); return obj,'serialized_module'
 except Exception: pass
 raise RuntimeError('Frozen Notebook46 U-Net is state_dict-only or uses a custom constructor. Copy the EXACT Notebook46 R3 model constructor into load_frozen_unet; Notebook51 deliberately refuses to invent an architecture.')
UNET,UNET_FORMAT=load_frozen_unet(UNET_CKPT)
def restore_unet(obs,mask):
 result=obs.copy();
 for x0,y0,x1,y1 in padded_components(mask):
  im=obs[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
  # Generic frozen-module contract: input channels [observed/255, mask]. Output one grayscale channel in [0,1] or [-1,1].
  x=np.stack([im.astype(np.float32)/255.,mk.astype(np.float32)])[None]
  with torch.no_grad(): y=UNET(torch.from_numpy(x).to(DEVICE))
  if isinstance(y,(tuple,list)): y=y[0]
  y=y.detach().float().cpu().numpy().squeeze()
  if y.min()<-0.05: y=(y+1)/2
  cand=np.clip(y*255,0,255).astype(np.uint8)
  require(cand.shape==im.shape,f'U-Net output shape {cand.shape} != crop {im.shape}')
  result[y0:y1,x0:x1][mk]=cand[mk]
 return immutable(obs,result,mask)
print('Frozen U-Net format:',UNET_FORMAT)


## 6. Build all candidates without loading clean truth; page-checkpointed and restartable


In [ ]:
def checkpoint_valid(page,contract):
 p=CHECK/f'{page}.json'
 if not p.exists(): return False
 try:
  rec=json.loads(p.read_text())
  if rec.get('contract')!=contract:return False
  return all(Path(x['path']).exists() and sha(Path(x['path']))==x['sha256'] for x in rec.get('outputs',[]))
 except Exception:return False
contract=hashlib.sha256(json.dumps({'pre':PREREG,'n50':sha(N50/'notebook50_verdict.json'),'n45':sha(N45/'mask_lock.json'),'n46':sha(N46/'experiment_identity.json'),'unet':sha(UNET_CKPT)},sort_keys=True).encode()).hexdigest()
completion=[]
for page in pages:
 if checkpoint_valid(page,contract): completion.append({'page':page,'status':'reused'}); continue
 obs=cv2.imread(str(observed_map[page]),0); require(obs is not None,f'{page}: observed unreadable')
 # authoritative mask from canonical V7.5 cache
 with np.load(cache_lookup[page]) as z:
  require('mask' in z.files,f'{page}: cache lacks authoritative mask'); auth=z['mask']>0
 require(auth.shape==obs.shape,f'{page}: auth shape mismatch')
 outputs=[]
 for arm in MASK_ARMS:
  mk=masks[page][arm]; require(mk.shape==obs.shape,f'{page}/{arm}: shape mismatch'); require(not np.any(mk&~auth),f'{page}/{arm}: active residual escapes authoritative mask')
  for r in RESTORERS:
   cand=restore_classical(obs,mk,r) if r!='R3_learned_unet' else restore_unet(obs,mk)
   require(np.array_equal(cand[~auth],obs[~auth]),f'{page}/{arm}/{r}: outside-authority drift')
   dest=CAND/page/arm; dest.mkdir(parents=True,exist_ok=True); path=dest/f'{r}.png'; cv2.imwrite(str(path),cand); outputs.append({'arm':arm,'restorer':r,'path':str(path),'sha256':sha(path),'array_sha256':ah(cand)})
 atomic_json(CHECK/f'{page}.json',{'page':page,'contract':contract,'outputs':outputs,'utc':datetime.now(timezone.utc).isoformat()}); completion.append({'page':page,'status':'computed'})
pd.DataFrame(completion).to_csv(AUDIT/'page_completion.csv',index=False)
print(pd.Series([x['status'] for x in completion]).value_counts())


## 7. Candidate lock, then clean-truth unlock


In [ ]:
lock_rows=[]
for page in pages:
 for arm in MASK_ARMS:
  for r in RESTORERS:
   p=CAND/page/arm/f'{r}.png'; require(p.exists(),f'Missing candidate {p}'); lock_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'arm':arm,'restorer':r,'path':str(p),'sha256':sha(p)})
lock_df=pd.DataFrame(lock_rows).sort_values(['page','arm','restorer']); lock_df.to_csv(LOCK/'candidate_inventory.csv',index=False)
lock_id=hashlib.sha256(lock_df.to_csv(index=False).encode()).hexdigest(); atomic_json(LOCK/'candidate_lock.json',{'lock_id':lock_id,'contract':contract,'rows':len(lock_df),'truth_loaded':False})
# Truth is opened only here.
clean_map=imap(ROOT/'source'/'clean'); require(set(pages).issubset(clean_map),f'Missing clean pages after lock: {sorted(set(pages)-set(clean_map))}')
print('Candidate lock:',lock_id,'Clean truth unlocked for scoring only.')


## 8. Whole-page and mask-region scoring


In [ ]:
def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge(im): return cv2.Canny(im,70,170)>0
def f1(ref,pred,region,tol=1):
 ref=ref&region; pred=pred&region
 if ref.sum()==0:return np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8); rd=cv2.dilate(ref.astype(np.uint8),k)>0; pdil=cv2.dilate(pred.astype(np.uint8),k)>0
 pr=np.sum(pred&rd)/max(pred.sum(),1); rc=np.sum(ref&pdil)/ref.sum(); return float(2*pr*rc/max(pr+rc,1e-12))
def score(clean,cand,region):
 diff=np.abs(clean.astype(np.float32)-cand.astype(np.float32)); ci=ink(clean); pi=ink(cand)
 return {'mae':float(diff[region].mean()) if region.any() else np.nan,'ssim':float(ssim(clean,cand,data_range=255)),'ink_f1':f1(ci,pi,region,1),'edge_f1':f1(edge(clean),edge(cand),region,1),'false_inserted_ink':int((pi&~ci&region).sum()),'missing_ink':int((ci&~pi&region).sum())}
rows=[]
for page in pages:
 clean=cv2.imread(str(clean_map[page]),0); obs=cv2.imread(str(observed_map[page]),0)
 with np.load(cache_lookup[page]) as z: auth=z['mask']>0
 for arm in MASK_ARMS:
  region=auth  # fixed scoring region avoids rewarding a smaller mask merely by excluding pixels from evaluation
  for r in RESTORERS:
   cand=cv2.imread(str(CAND/page/arm/f'{r}.png'),0); q=score(clean,cand,region); rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'arm':arm,'restorer':r,'scoring_pixels':int(region.sum()),**q})
metrics=pd.DataFrame(rows); metrics.to_csv(TABLES/'candidate_metrics.csv',index=False)
summary=metrics.groupby(['arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ssim=('ssim','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'))
summary.to_csv(TABLES/'candidate_summary.csv',index=False); display(summary)


## 9. Paired M5-REMOVE versus M3 page-clustered bootstrap


In [ ]:
def benefit_delta(metric,m5,m3):
 # positive always means M5 is better
 return (m3-m5) if metric in ['mae','false_inserted_ink','missing_ink'] else (m5-m3)
def boot_ci(v,n=BOOTSTRAP,seed=SEED):
 v=np.asarray(v,float); v=v[np.isfinite(v)]; rng=np.random.default_rng(seed)
 if not len(v):return (np.nan,np.nan,np.nan)
 b=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(n)]); return float(v.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))
paired=[]
for r in RESTORERS:
 q=metrics[metrics.restorer==r]
 for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
  p=q.pivot(index='page',columns='arm',values=metric); d=benefit_delta(metric,p['M5_REMOVE'],p['M3']); mean,lo,hi=boot_ci(d)
  paired.append({'restorer':r,'metric':metric,'benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum()),'pages':int(d.notna().sum())})
paired_df=pd.DataFrame(paired); paired_df.to_csv(TABLES/'paired_m5_vs_m3_bootstrap.csv',index=False); display(paired_df)


## 10. Restorer-gap contraction

For each page and mask arm, define the restorer gap as the dispersion across the three frozen restorers. We report both range and population standard deviation for MAE, ink F1, edge F1, false inserted ink and missing ink. A positive contraction means the M3 gap is larger than the M5-REMOVE gap. This endpoint is secondary: contraction is favourable only if M5 does not make all restorers uniformly worse.


In [ ]:
gap_rows=[]
for (page,arm),g in metrics.groupby(['page','arm']):
 row={'page':page,'source_group':g.source_group.iloc[0],'arm':arm}
 for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
  v=g[metric].to_numpy(float); row[f'{metric}_range']=float(np.nanmax(v)-np.nanmin(v)); row[f'{metric}_sd']=float(np.nanstd(v))
 gap_rows.append(row)
gaps=pd.DataFrame(gap_rows); gaps.to_csv(TABLES/'restorer_gap_by_page.csv',index=False)
contract_rows=[]
for metric in ['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
 for stat in ['range','sd']:
  p=gaps.pivot(index='page',columns='arm',values=f'{metric}_{stat}'); d=p['M3']-p['M5_REMOVE']; mean,lo,hi=boot_ci(d)
  contract_rows.append({'metric':metric,'gap_stat':stat,'contraction_mean':mean,'ci95_low':lo,'ci95_high':hi,'pages_contracting':int((d>0).sum()),'pages_expanding':int((d<0).sum())})
contraction=pd.DataFrame(contract_rows); contraction.to_csv(TABLES/'restorer_gap_contraction.csv',index=False); display(contraction)
# Source-group paired summaries
source=[]
for (sg,r),g in metrics.groupby(['source_group','restorer']):
 for metric in ['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_ink']:
  p=g.pivot(index='page',columns='arm',values=metric); d=benefit_delta(metric,p['M5_REMOVE'],p['M3']); source.append({'source_group':sg,'restorer':r,'metric':metric,'pages':len(d),'benefit_mean':float(d.mean()),'wins':int((d>0).sum()),'losses':int((d<0).sum())})
pd.DataFrame(source).to_csv(TABLES/'m5_vs_m3_by_source_group.csv',index=False)


## 11. Registered interpretation and verdict


In [ ]:
# Primary requirement: no integrity failure and evidence of non-negative structural movement.
p=paired_df.set_index(['restorer','metric'])
primary_edge=p.loc[(PRIMARY_RESTORER,'edge_f1')]; primary_ink=p.loc[(PRIMARY_RESTORER,'ink_f1')]; primary_mae=p.loc[(PRIMARY_RESTORER,'mae')]
# Conservative causal evidence: lower CI >0 on at least one structural endpoint for >=2 restorers, and no restorer structural CI upper <0.
struct=paired_df[paired_df.metric.isin(['ink_f1','edge_f1'])].copy()
positive_restorers=set(struct.loc[struct.ci95_low>0,'restorer']); harmful=struct[struct.ci95_high<0]
integrity=True
causal=bool(len(positive_restorers)>=2 and harmful.empty)
contraction_positive=int((contraction.ci95_low>0).sum())
if not integrity: verdict='INVALID_RUN'
elif causal and contraction_positive>0: verdict='M5_REMOVE_DEMONSTRATES_CAUSAL_RECONSTRUCTION_GAIN_WITH_SOME_GAP_CONTRACTION'
elif causal: verdict='M5_REMOVE_DEMONSTRATES_CAUSAL_RECONSTRUCTION_GAIN_NO_ROBUST_GAP_CONTRACTION'
elif harmful.empty: verdict='M5_REMOVE_RESEARCH_SIGNAL_NOT_DEMONSTRATED'
else: verdict='M5_REMOVE_RECONSTRUCTION_REGRESSION'
record={'verdict':verdict,'development_only':True,'promotion_authority':False,'positive_structural_restorers':sorted(positive_restorers),'structural_harm_rows':harmful.to_dict('records'),'robust_gap_contraction_endpoints':contraction_positive,'primary_restorer':PRIMARY_RESTORER,'m4_role':'diagnostic_only'}
atomic_json(OUT/'notebook51_verdict.json',record); print(json.dumps(record,indent=2))


## 12. Visual review queues and contact sheets


In [ ]:
# Rank pages by M5 benefit/regression under each restorer using edge F1 and false/missing trade-off.
queue=[]
for r in RESTORERS:
 q=metrics[metrics.restorer==r].set_index(['page','arm'])
 for page in pages:
  a=q.loc[(page,'M3')]; b=q.loc[(page,'M5_REMOVE')]
  queue.append({'page':page,'source_group':a.source_group,'restorer':r,'edge_f1_benefit':b.edge_f1-a.edge_f1,'ink_f1_benefit':b.ink_f1-a.ink_f1,'mae_benefit':a.mae-b.mae,'false_inserted_benefit':a.false_inserted_ink-b.false_inserted_ink,'missing_ink_benefit':a.missing_ink-b.missing_ink})
queue=pd.DataFrame(queue); queue.to_csv(TABLES/'visual_audit_queue.csv',index=False)
for r in RESTORERS:
 for label,q in [('best',queue[queue.restorer==r].sort_values(['edge_f1_benefit','ink_f1_benefit'],ascending=False).head(12)),('worst',queue[queue.restorer==r].sort_values(['edge_f1_benefit','ink_f1_benefit']).head(12))]:
  if q.empty: continue
  fig,axes=plt.subplots(len(q),3,figsize=(15,5*len(q)),squeeze=False)
  for i,row in enumerate(q.itertuples()):
   for j,(arm,title) in enumerate([('M3','M3'),('M5_REMOVE','M5-REMOVE'),('M4_DIAGNOSTIC','M4 diagnostic')]):
    im=cv2.imread(str(CAND/row.page/arm/f'{r}.png'),0); axes[i,j].imshow(im,cmap='gray',vmin=0,vmax=255); axes[i,j].set_title(f'{row.page} {title}'); axes[i,j].axis('off')
  plt.tight_layout(); plt.savefig(VIS/f'{r}_{label}_pages.png',dpi=120,bbox_inches='tight'); plt.close(fig)
print('Visual audit:',VIS)


## 13. Experiment identity and curated Final Upload Package


In [ ]:
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'device':DEVICE,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; atomic_json(OUT/'software_environment.json',env)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook51-frozen-mask-restorer-crossover','source_record':source,'candidate_lock_id':lock_id,'contract_sha256':contract,'mask_arms':MASK_ARMS,'restorers':RESTORERS,'unet_checkpoint':str(UNET_CKPT),'unet_checkpoint_sha256':sha(UNET_CKPT),'unet_format':UNET_FORMAT,'pages':len(pages),'source_groups':int(meta.loc[pages].source_group.nunique()),'software':env,'development_only':True,'promotion_authority':False}; atomic_json(OUT/'experiment_identity.json',identity)
lines=['V8.7 NOTEBOOK 51 - FROZEN M3 / M5-REMOVE / DIAGNOSTIC-M4 CROSSOVER','',f'Run: {OUT}',f'Pages: {len(pages)}',f'Source groups: {identity["source_groups"]}',f'Verdict: {verdict}','',f'Positive structural restorers: {sorted(positive_restorers)}',f'Robust gap-contraction endpoints: {contraction_positive}','','No PROTECT intervention. M4 diagnostic only. Clean truth scoring-only. No production authority.']
(OUT/'executive_summary.txt').write_text('
'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook51_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',OUT/'01_pre_registration.json',LOCK/'candidate_lock.json',TABLES/'mask_arm_inventory.csv',TABLES/'candidate_summary.csv',TABLES/'candidate_metrics.csv',TABLES/'paired_m5_vs_m3_bootstrap.csv',TABLES/'restorer_gap_contraction.csv',TABLES/'m5_vs_m3_by_source_group.csv',TABLES/'visual_audit_queue.csv',AUDIT/'page_completion.csv']
for p in required: require(p.exists(),f'Missing upload artefact {p}')
for p in required:
 dst=UPLOAD/p.name; shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 51 curated review package. Candidate images and page checkpoints remain in the run directory and are identified by experiment_identity.json.' + chr(10))
rows=[]
for p in sorted(UPLOAD.iterdir()):
 if p.is_file(): rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=UPLOAD/'notebook51_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(UPLOAD.iterdir()):
  if p.is_file() and p!=zip_path: z.write(p,p.name)
print('
'.join(lines)); print('Bundle:',zip_path)


## Review order
1. `executive_summary.txt`
2. `notebook51_verdict.json`
3. `05_tables/paired_m5_vs_m3_bootstrap.csv`
4. `05_tables/restorer_gap_contraction.csv`
5. `05_tables/m5_vs_m3_by_source_group.csv`
6. `05_tables/candidate_summary.csv`
7. `06_visual_audit/`
8. `upload/notebook51_upload_bundle.zip`

If the frozen U-Net is stored as a state dictionary rather than TorchScript/serialized module, Notebook 51 will stop in Cell 5 rather than guess the architecture. In that case copy the exact R3 model constructor and preprocessing contract from Notebook 46 into that cell, without changing weights or inference semantics.
